# 1.3 — Statistics & Probability
**Distributions, Mean/Std, Bayes' Theorem**

> Core idea: Before building any model, you need to understand your data's shape. Statistics tells you what's normal, what's an outlier, and how confident you should be.

## Why ML needs Statistics

When you get a new dataset, before writing a single line of ML code you need to answer:
- What's the **typical** value for each feature?
- How **spread out** are the values?
- Is the data **skewed** or symmetric?
- Are there **outliers** that will confuse the model?
- How **likely** is each outcome?

All of this is statistics. Skip it and your model will train on garbage.

## Part 1 — Descriptive Statistics

### Mean, Median, Mode

| Measure | Formula | When to use |
|---|---|---|
| **Mean** | sum / count | Normal data (no outliers) |
| **Median** | middle value | Skewed data (salaries, prices) |
| **Mode** | most frequent | Categorical data |

**Analogy:** 5 friends in a room earning ₹30k/month each. Average = ₹30k. Now one friend earns ₹10 crore. Average shoots to ₹2cr — misleading! Median stays near ₹30k — much more honest.

### Standard Deviation & Variance

```
Variance  = average of (each value - mean)²
Std Dev   = √variance   ← same units as original data
```

Std Dev = how far values typically stray from the mean.
- Small std dev → values clustered tightly
- Large std dev → values spread widely

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Salaries in a Kochi office (₹ per month)
salaries = np.array([25000, 28000, 30000, 32000, 31000, 29000, 27000, 500000])  # last = CEO

print("--- Measures of Central Tendency ---")
print(f"Mean:   ₹{np.mean(salaries):,.0f}")
print(f"Median: ₹{np.median(salaries):,.0f}")
print(f"Mode:   ₹{stats.mode(salaries, keepdims=True).mode[0]:,.0f}")

print("\n--- Spread ---")
print(f"Std Dev:  ₹{np.std(salaries):,.0f}")
print(f"Variance: {np.var(salaries):,.0f}")
print(f"Min: ₹{salaries.min():,} | Max: ₹{salaries.max():,}")

## Part 2 — Distributions

### Normal Distribution (Bell Curve)
- Symmetric around the mean
- 68% of data within ±1 std dev
- 95% within ±2 std devs
- 99.7% within ±3 std devs

**Examples:** Heights, exam scores, measurement errors

### Skewed Distribution
- **Right skew (positive):** long tail to the right. Mean > Median. Example: salaries, house prices
- **Left skew (negative):** long tail to the left. Mean < Median. Example: age at retirement

**Why it matters for ML:** Use median (not mean) to fill missing values in skewed data. Use IQR (not Z-score) to detect outliers.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Normal distribution
normal_data = np.random.normal(loc=50, scale=10, size=1000)
axes[0].hist(normal_data, bins=30, color='steelblue', edgecolor='white')
axes[0].axvline(np.mean(normal_data), color='red', label='Mean')
axes[0].axvline(np.median(normal_data), color='green', linestyle='--', label='Median')
axes[0].set_title("Normal — mean ≈ median")
axes[0].legend(fontsize=9)

# Right skewed
skewed_data = np.random.exponential(scale=20000, size=1000) + 20000
axes[1].hist(skewed_data, bins=30, color='coral', edgecolor='white')
axes[1].axvline(np.mean(skewed_data), color='red', label='Mean')
axes[1].axvline(np.median(skewed_data), color='green', linestyle='--', label='Median')
axes[1].set_title("Right skew — mean > median (salaries)")
axes[1].legend(fontsize=9)

# After log transform
log_data = np.log1p(skewed_data)
axes[2].hist(log_data, bins=30, color='mediumseagreen', edgecolor='white')
axes[2].set_title("After log transform — more normal")

plt.tight_layout()
plt.show()

## Part 3 — Probability Basics

### What is probability?
A number between 0 and 1 expressing how likely an event is.
- 0 = impossible
- 1 = certain
- 0.5 = 50/50

### Conditional Probability
P(A | B) = probability of A **given** B has happened

Example: P(loan default | age < 25) — probability of defaulting, given the person is under 25.

### Bayes' Theorem
```
P(A|B) = P(B|A) × P(A) / P(B)
```

**Plain English:** Update your belief about A after seeing evidence B.

**Analogy:** You think 10% of emails are spam (prior). You see the word "FREE MONEY" — that updates your belief to 90% spam (posterior). Bayes is that update formula.

| Term | Meaning | Example |
|---|---|---|
| P(A) | Prior — belief before evidence | 10% of emails are spam |
| P(B\|A) | Likelihood — how likely is evidence given A | "FREE MONEY" appears in 80% of spam |
| P(A\|B) | Posterior — updated belief after evidence | Given "FREE MONEY", 90% chance it's spam |

In [ ]:
# Bayes Theorem — Manual calculation
# A medical test for a rare disease

P_disease = 0.001        # 0.1% of population has the disease
P_positive_given_disease = 0.99   # test catches 99% of true cases
P_positive_given_no_disease = 0.05  # 5% false positive rate

P_no_disease = 1 - P_disease

# P(positive) = total probability of testing positive
P_positive = (P_positive_given_disease * P_disease + 
              P_positive_given_no_disease * P_no_disease)

# Bayes: P(disease | positive test)
P_disease_given_positive = (P_positive_given_disease * P_disease) / P_positive

print(f"P(disease)          = {P_disease:.3f}  ({P_disease*100:.1f}%)")
print(f"P(positive | sick)  = {P_positive_given_disease:.3f}  ({P_positive_given_disease*100:.1f}%)")
print(f"P(positive | well)  = {P_positive_given_no_disease:.3f}  ({P_positive_given_no_disease*100:.1f}%)")
print(f"\nP(sick | positive)  = {P_disease_given_positive:.3f}  ({P_disease_given_positive*100:.1f}%)")
print("\nSurprising result: even with a 99% accurate test, a positive result")
print("only means ~2% chance of actually having the disease (because disease is rare).")

## Part 4 — Correlation

Correlation measures how two features move together.
- Range: -1 to +1
- +1 = perfectly together (height and weight)
- -1 = perfectly opposite (temperature and heating bill)
- 0 = no relationship

**Important:** Correlation ≠ Causation. Ice cream sales and drowning rates are correlated (both peak in summer) — ice cream doesn't cause drowning.

In [ ]:
import pandas as pd

# House prices in Kerala
np.random.seed(42)
n = 100
size = np.random.normal(1500, 300, n)          # sqft
price = size * 3000 + np.random.normal(0, 200000, n)  # ₹
age = np.random.randint(1, 30, n)              # years old
bedrooms = np.random.randint(1, 5, n)

df = pd.DataFrame({'size_sqft': size, 'price': price, 'age_years': age, 'bedrooms': bedrooms})

# Correlation with price
print("Correlation with house price:")
print(df.corr()['price'].sort_values(ascending=False))

# This tells you which features are most useful for predicting price

## Summary

| Concept | Key point | ML use |
|---|---|---|
| Mean | Average — sensitive to outliers | Fill missing values (normal data) |
| Median | Middle value — robust | Fill missing values (skewed data) |
| Std Dev | Spread of data | StandardScaler divides by this |
| Normal dist. | Bell curve — 68/95/99.7 rule | Z-score outlier detection |
| Skewed dist. | Long tail — mean misleads | Use IQR for outliers, log transform |
| Bayes | Update belief with evidence | Naive Bayes classifier (Ch 4.6) |
| Correlation | -1 to +1 — feature relationships | Feature selection in EDA |

## Practice Task

Auto prices dataset (made up, Kochi dealers):

In [ ]:
car_prices = np.array([450000, 480000, 510000, 490000, 520000, 475000, 2500000, 460000])
# Last one is a luxury import

# YOUR CODE HERE
# Q1: What is the mean and median price?

# Q2: Which is more representative of a typical car price here?

# Q3: What is the standard deviation?

# Q4: Is this data right-skewed or normal? How do you know?